# 02. Diabetes Health Indicators: Model Training and Evaluation

**Objective:** To build, train, and evaluate machine learning pipelines using the preprocessed BRFSS dataset, with a strong focus on handling class imbalance using appropriate metrics (F1-Score, Recall).

In [3]:
import pandas as pd
import numpy as np

# Load the preprocessed and split datasets
X_train = pd.read_csv('datasets/X_train.csv')
X_test = pd.read_csv('datasets/X_test.csv')
y_train = pd.read_csv('datasets/y_train.csv').squeeze() # squeeze() converts DataFrame to Series
y_test = pd.read_csv('datasets/y_test.csv').squeeze()

print('Data successfully loaded')
print(f'X_train shape: {X_train.shape}')
print(f'y_train shape: {y_train.shape}')

Data successfully loaded
X_train shape: (183824, 21)
y_train shape: (183824,)


## 1. Preprocessing Pipeline Setup
Before feeding the data into a machine learning model, we must preprocess our features based on their inherent data types and distributions.

1. **Log Transformation for Skewed Data (`BMI`):**
   Our EDA revealed that `BMI` is highly right-skewed with extreme values (>70). To prevent these outliers from disproportionately influencing the Logistic Regression decision boundary, we apply a Log Transformation (`np.log1p`) followed by `StandardScaler`.
2. **Standard Scaling for Ordinal Variables (`Age`, `Income`, etc.):**
   Variables like `Age` and `Income` are ordinal categories. We treat them as continuous and apply `StandardScaler`.
3. **Passthrough for Binary Variables (`Sex`, `HighBP`, etc.):**
   These are already encoded as 0 or 1, requiring no further transformation.

In [20]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, FunctionTransformer
from sklearn.pipeline import Pipeline

# Grouping features based on their types
skewed_features = ['BMI']
num_ord_features = ['GenHlth', 'MentHlth', 'PhysHlth', 'Age', 'Education', 'Income']
binary_features = ['HighBP', 'HighChol', 'CholCheck', 'Smoker', 'Stroke', 'HeartDiseaseorAttack', 'PhysActivity', 'Fruits', 
                   'Veggies', 'HvyAlcoholConsump', 'AnyHealthcare', 'NoDocbcCost', 'DiffWalk', 'Sex']

# Custom pipeline for skewed data (Log transform -> Standard Scale)
# We use np.log1p (log(1+x)) which is mathematically safer
log_scale_transformer = Pipeline(steps=[
    ('log', FunctionTransformer(np.log1p)),
    ('scaler', StandardScaler())
])

# Building the final preprocessor
preprocessor = ColumnTransformer(
    transformers=[
        ('skewed', log_scale_transformer, skewed_features),
        ('num_ord', StandardScaler(), num_ord_features),
        ('binary', 'passthrough', binary_features)
    ])

print('Preprocessing pipeline successfully configured.')

Preprocessing pipeline successfully configured.
